---

# Chapter 1 — Data Importation, Preparation & Validation

---

## 1.0 Environment Setup

All libraries are imported here. 



In [2]:
# ═══════════════════════════════════════════════════════════════
# CHAPTER 1.0 — LIBRARY IMPORTS
# All libraries needed for Chapter 0 and Chapter 1 only.
# ═══════════════════════════════════════════════════════════════

# ── Core ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import os, sys

# ── Ensure src/ is on the path ────────────────────────────────
# If notebook is opened from notebooks/ subfolder, add project root
sys.path.insert(0, os.path.abspath('..'))

# ── src modules ───────────────────────────────────────────────
from src.config      import *          # paths, palette, constants, plot style
from src.data_loader import load_raw, load_processed
from src.features    import get_feature_summary

# ── Display ───────────────────────────────────────────────────
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print(" Chapter 1 libraries loaded")
print(f"  Project root : {PROJECT_ROOT}")
print(f"  Raw data     : {DATA_RAW}")
print(f"  Reports dir  : {REPORTS_DIR}")


 config loaded | project root: C:\Users\Administrator\Desktop\insurance-claim-fraud-detection
 Chapter 1 libraries loaded
  Project root : C:\Users\Administrator\Desktop\insurance-claim-fraud-detection
  Raw data     : C:\Users\Administrator\Desktop\insurance-claim-fraud-detection\data\insurance_claims.csv
  Reports dir  : C:\Users\Administrator\Desktop\insurance-claim-fraud-detection\reports


---

## 1.1 Load Raw Data (Unprocessed)

First load the raw CSV exactly as it, no transformations.
This is for inspection only. All downstream chapters use `load_processed()`.


In [3]:
# ═══════════════════════════════════════════════════════════════
# CHAPTER 1.1 — LOAD RAW DATA
# ═══════════════════════════════════════════════════════════════

# data path
DATA_RAW = r"C:\Users\Administrator\Desktop\insurance-claim-fraud-detection\data\insurance_claims.csv"
df_raw = pd.read_csv(DATA_RAW, low_memory=False)

print()
print("First 3 rows:")
df_raw.head(3)



First 3 rows:


,TXN_DATE_TIME,TRANSACTION_ID,CUSTOMER_ID,POLICY_NUMBER,POLICY_EFF_DT,LOSS_DT,REPORT_DT,INSURANCE_TYPE,PREMIUM_AMOUNT,CLAIM_AMOUNT,CUSTOMER_NAME,ADDRESS_LINE1,ADDRESS_LINE2,CITY,STATE,POSTAL_CODE,SSN,MARITAL_STATUS,AGE,TENURE,EMPLOYMENT_STATUS,NO_OF_FAMILY_MEMBERS,RISK_SEGMENTATION,HOUSE_TYPE,SOCIAL_CLASS,ROUTING_NUMBER,ACCT_NUMBER,CUSTOMER_EDUCATION_LEVEL,CLAIM_STATUS,INCIDENT_SEVERITY,AUTHORITY_CONTACTED,ANY_INJURY,POLICE_REPORT_AVAILABLE,INCIDENT_STATE,INCIDENT_CITY,INCIDENT_HOUR_OF_THE_DAY,AGENT_ID,VENDOR_ID
0,2020-06-01 00:00:00,TXN00000001,A00003822,PLC00008468,2015-06-23,2020-05-16,2020-05-21,Health,157.13,9000,Christopher Demarest,7701 West Saint John Road,#2010,Glendale,AZ,85308,087-11-1946,Y,54,89,Y,3,L,Own,LI,109134974,HXJP58258181908465,Bachelor,A,Major Loss,Police,0,1,GA,Savannah,4,AGENT00413,VNDR00556
1,2020-06-01 00:00:00,TXN00000002,A00008149,PLC00009594,2018-04-21,2020-05-13,2020-05-18,Property,141.71,26000,Ricardo Gatlin,8595 West 81st Drive,NaN,Arvada,CO,80005,685-33-3536,N,61,80,Y,4,L,Rent,MI,40125819,JUND46859540983731,Bachelor,A,Total Loss,Ambulance,1,0,AL,Montgomery,0,AGENT00769,VNDR00592
2,2020-06-01 00:00:00,TXN00000003,A00003172,PLC00007969,2019-10-03,2020-05-21,2020-05-26,Property,157.24,13000,Lashawn Engles,637 Britannia Drive,NaN,Vallejo,CA,94591,378-36-0672,N,47,68,Y,6,L,Rent,MI,99513168,WGZZ90128415227650,PhD,A,Total Loss,Police,0,1,CO,Grand Junction,19,AGENT00883,VNDR00031


---

## 1.2 Data Types & Column Inspection

Before any transformation, document what Python infers about each column.
This forms the basis of the **Data Description** subsection of the research report.


In [4]:
# ═══════════════════════════════════════════════════════════════
# CHAPTER 1.2 — DTYPE & NULL INSPECTION
# ═══════════════════════════════════════════════════════════════

print(f"{'Column':<32} {'Dtype':<12} {'Non-Null':>10} {'Null':>8} {'Null%':>8}")
print("─" * 76)
for col in df_raw.columns:
    n_null = df_raw[col].isnull().sum()
    n_ok   = df_raw[col].notnull().sum()
    pct    = n_null / len(df_raw) * 100
    print(f"  {col:<30} {str(df_raw[col].dtype):<12} {n_ok:>10,} {n_null:>8,} {pct:>7.1f}%")


Column                           Dtype          Non-Null     Null    Null%
────────────────────────────────────────────────────────────────────────────
  TXN_DATE_TIME                  object           10,000        0     0.0%
  TRANSACTION_ID                 object           10,000        0     0.0%
  CUSTOMER_ID                    object           10,000        0     0.0%
  POLICY_NUMBER                  object           10,000        0     0.0%
  POLICY_EFF_DT                  object           10,000        0     0.0%
  LOSS_DT                        object           10,000        0     0.0%
  REPORT_DT                      object           10,000        0     0.0%
  INSURANCE_TYPE                 object           10,000        0     0.0%
  PREMIUM_AMOUNT                 float64          10,000        0     0.0%
  CLAIM_AMOUNT                   int64            10,000        0     0.0%
  CUSTOMER_NAME                  object           10,000        0     0.0%
  ADDRESS_LINE1        

---

## 1.3 Load Processed Data

`load_processed()` from `src/data_loader.py` applies all transformations in one call:
date casting, target variable creation, lag feature engineering, label columns.
Every chapter that needs data calls this same function — ensuring consistency.


In [5]:
# ═══════════════════════════════════════════════════════════════
# CHAPTER 1.3 — LOAD PROCESSED DATA
# src/data_loader.load_processed() applies:
#   - Date casting (4 columns)
#   - IS_DENIED binary target
#   - Lag features: REPORT_LAG_DAYS, PROCESS_LAG_DAYS, TOTAL_LAG_DAYS
#   - POLICY_AGE_AT_LOSS_DAYS
#   - VENDOR_INVOLVED flag
#   - CUSTOMER_EDUCATION_LEVEL nulls → 'Unknown'
#   - Readable label columns for plots
# ═══════════════════════════════════════════════════════════════

df = load_processed(verbose=True)

print()
print(f"New engineered columns added:")
new_cols = [
    'IS_DENIED', 'REPORT_LAG_DAYS', 'PROCESS_LAG_DAYS',
    'TOTAL_LAG_DAYS', 'POLICY_AGE_AT_LOSS_DAYS', 'VENDOR_INVOLVED',
    'STATUS_LABEL', 'RISK_LABEL', 'MARITAL_LABEL', 'EMPLOYMENT_LABEL', 'CLASS_LABEL'
]
for col in new_cols:
    sample = df[col].iloc[0]
    dtype  = str(df[col].dtype)
    print(f"  {col:<30} {dtype:<12} sample: {sample}")


 Processed data loaded: 10,000 rows × 49 columns
  IS_DENIED: 503 denied (5.03%)

New engineered columns added:
  IS_DENIED                      int64        sample: 0
  REPORT_LAG_DAYS                int64        sample: 5
  PROCESS_LAG_DAYS               int64        sample: 11
  TOTAL_LAG_DAYS                 int64        sample: 16
  POLICY_AGE_AT_LOSS_DAYS        int64        sample: 1789
  VENDOR_INVOLVED                int64        sample: 1
  STATUS_LABEL                   object       sample: Approved
  RISK_LABEL                     object       sample: Low
  MARITAL_LABEL                  object       sample: Married
  EMPLOYMENT_LABEL               object       sample: Employed
  CLASS_LABEL                    object       sample: Low Income


### 

---

## 1.4 Data Quality Validation

Cross checking every SQL audit finding in Python with pass/fail assertions.
This section is the **Data Quality** subsection of the methodology chapter.


In [7]:
# ═══════════════════════════════════════════════════════════════
# CHAPTER 1.4 — DATA QUALITY VALIDATION
# Replicate SQL audit findings in Python
# ═══════════════════════════════════════════════════════════════

print("=" * 58)
print("  DATA QUALITY REPORT")
print("=" * 58)

# Row count
print(f"\n[1] Row count    : {len(df):,}  (Expected: {TOTAL_ROWS:,})")
assert len(df) == TOTAL_ROWS, "Row count mismatch!"

# Unique key checks
print(f"[2] Unique IDs:")
print(f"    TRANSACTION_ID : {df['TRANSACTION_ID'].nunique():,}  (Expected: 10,000)")
print(f"    CUSTOMER_ID    : {df['CUSTOMER_ID'].nunique():,}  (Expected: 10,000)")
print(f"    POLICY_NUMBER  : {df['POLICY_NUMBER'].nunique():,}  (Expected: 10,000)")

# Null audit vs SQL findings
print("\n[3] Null audit (vs SQL baseline):")
null_expected = {
    'ADDRESS_LINE2'           : 8505,
    'VENDOR_ID'               : 3245,
    'AUTHORITY_CONTACTED'     : 1945,
    'CUSTOMER_EDUCATION_LEVEL': 0,    # filled → 'Unknown'
    'INCIDENT_CITY'           : 46,
    'CITY'                    : 54,
}
for col, exp in null_expected.items():
    # Use df_raw for columns we didn't fill
    src_df = df_raw if col in ['AUTHORITY_CONTACTED','VENDOR_ID',
                                 'ADDRESS_LINE2','INCIDENT_CITY','CITY'] else df
    actual = src_df[col].isnull().sum()
    status = "✓" if actual == exp else "MISMATCH"
    label  = " (filled → 'Unknown')" if col == 'CUSTOMER_EDUCATION_LEVEL' else ""
    print(f"    {col:<30}: {actual:,}  (exp {exp:,}) {status}{label}")

# Target distribution
print(f"\n[4] Target variable (CLAIM_STATUS):")
for val, cnt in df['CLAIM_STATUS'].value_counts().items():
    lbl = 'Approved' if val == 'A' else 'Denied (fraud proxy)'
    print(f"    {val} ({lbl:<23}): {cnt:,}  ({cnt/len(df)*100:.2f}%)")

# Numeric ranges
print("\n[5] Numeric ranges:")
checks = [
    ('PREMIUM_AMOUNT', 6, 200),
    ('CLAIM_AMOUNT', 100, 100000),
    ('AGE', 25, 64),
    ('TENURE', 6, 119),
    ('INCIDENT_HOUR_OF_THE_DAY', 0, 23),
    ('REPORT_LAG_DAYS', 0, 5),
    ('PROCESS_LAG_DAYS', 1, 15),
]
for col, lo, hi in checks:
    mn, mx = df[col].min(), df[col].max()
    ok = "✓" if mn >= lo and mx <= hi else "✗"
    print(f"    {col:<32}: min={mn}  max={mx}  {ok}")

print("\n" + "=" * 58)
print("All checks passed. Data ready for analysis.")
print("=" * 58)


  DATA QUALITY REPORT

[1] Row count    : 10,000  (Expected: 10,000)
[2] Unique IDs:
    TRANSACTION_ID : 10,000  (Expected: 10,000)
    CUSTOMER_ID    : 10,000  (Expected: 10,000)
    POLICY_NUMBER  : 10,000  (Expected: 10,000)

[3] Null audit (vs SQL baseline):
    ADDRESS_LINE2                 : 8,505  (exp 8,505) ✓
    VENDOR_ID                     : 3,245  (exp 3,245) ✓
    AUTHORITY_CONTACTED           : 1,945  (exp 1,945) ✓
    CUSTOMER_EDUCATION_LEVEL      : 0  (exp 0) ✓ (filled → 'Unknown')
    INCIDENT_CITY                 : 46  (exp 46) ✓
    CITY                          : 54  (exp 54) ✓

[4] Target variable (CLAIM_STATUS):
    A (Approved               ): 9,497  (94.97%)
    D (Denied (fraud proxy)   ): 503  (5.03%)

[5] Numeric ranges:
    PREMIUM_AMOUNT                  : min=6.0  max=200.0  ✓
    CLAIM_AMOUNT                    : min=100  max=100000  ✓
    AGE                             : min=25  max=64  ✓
    TENURE                          : min=6  max=119  ✓
    INC

---

## 1.5 Model Feature Summary

Documents all features that will enter the fraud model in Chapter 5.
This is **Table 1** in the research report.


In [8]:
# ═══════════════════════════════════════════════════════════════
# CHAPTER 1.5 — FEATURE SUMMARY (from src/features.py)
# ═══════════════════════════════════════════════════════════════

feature_table = get_feature_summary(df)
print("TABLE 1: Model Feature Inventory")
print("=" * 70)
print(feature_table.to_string(index=False))


TABLE 1: Model Feature Inventory
                 Feature        Type  Nulls  Unique     Sample
          PREMIUM_AMOUNT     Numeric      0    6550     157.13
            CLAIM_AMOUNT     Numeric      0     107       9000
                     AGE     Numeric      0      40         54
                  TENURE     Numeric      0     114         89
    NO_OF_FAMILY_MEMBERS     Numeric      0       6          3
              ANY_INJURY     Numeric      0       2          0
 POLICE_REPORT_AVAILABLE     Numeric      0       2          1
INCIDENT_HOUR_OF_THE_DAY     Numeric      0      24          4
         REPORT_LAG_DAYS     Numeric      0       5          5
        PROCESS_LAG_DAYS     Numeric      0      15         11
 POLICY_AGE_AT_LOSS_DAYS     Numeric      0    3474       1789
         VENDOR_INVOLVED     Numeric      0       2          1
          INSURANCE_TYPE Categorical      0       6     Health
       RISK_SEGMENTATION Categorical      0       3          L
          MARITAL_STAT

---

## 1.6 Descriptive Statistics

Tables 2 and 3 in the research report.


In [9]:
# ═══════════════════════════════════════════════════════════════
# CHAPTER 1.6 — DESCRIPTIVE STATISTICS
# ═══════════════════════════════════════════════════════════════

num_cols = [
    'PREMIUM_AMOUNT', 'CLAIM_AMOUNT', 'AGE', 'TENURE',
    'NO_OF_FAMILY_MEMBERS', 'INCIDENT_HOUR_OF_THE_DAY',
    'REPORT_LAG_DAYS', 'PROCESS_LAG_DAYS', 'POLICY_AGE_AT_LOSS_DAYS'
]

print("TABLE 2: Descriptive Statistics — Continuous Variables")
print("=" * 72)
desc = df[num_cols].describe().T
desc.columns = ['N','Mean','Std','Min','Q1','Median','Q3','Max']
desc['N'] = desc['N'].astype(int)
print(desc.round(2).to_string())

print()
print("TABLE 3: Categorical Variable Distributions")
print("=" * 72)
cat_cols = {
    'INSURANCE_TYPE'           : 'Insurance Type',
    'RISK_LABEL'               : 'Risk Segment',
    'CLASS_LABEL'              : 'Social Class',
    'HOUSE_TYPE'               : 'House Type',
    'MARITAL_LABEL'            : 'Marital Status',
    'EMPLOYMENT_LABEL'         : 'Employment Status',
    'CUSTOMER_EDUCATION_LEVEL' : 'Education Level',
    'INCIDENT_SEVERITY'        : 'Incident Severity',
    'STATUS_LABEL'             : 'Claim Status (Target)',
}
for col, label in cat_cols.items():
    counts = df[col].value_counts()
    parts  = [f"{v}: {n:,} ({n/len(df)*100:.1f}%)" for v, n in counts.items()]
    print(f"  {label:<30} → {' | '.join(parts)}")
    

TABLE 2: Descriptive Statistics — Continuous Variables
                              N      Mean       Std    Min       Q1   Median        Q3        Max
PREMIUM_AMOUNT            10000     88.51     48.32   6.00    62.02    88.84    121.90     200.00
CLAIM_AMOUNT              10000 16,563.83 22,037.49 100.00 2,000.00 7,000.00 21,000.00 100,000.00
AGE                       10000     44.47     11.52  25.00    34.00    45.00     54.00      64.00
TENURE                    10000     62.95     32.89   6.00    34.00    64.00     92.00     119.00
NO_OF_FAMILY_MEMBERS      10000      4.49      1.71   2.00     3.00     4.00      6.00       7.00
INCIDENT_HOUR_OF_THE_DAY  10000     11.60      6.88   0.00     6.00    12.00     18.00      23.00
REPORT_LAG_DAYS           10000      3.21      1.97   0.00     1.00     3.00      5.00       5.00
PROCESS_LAG_DAYS          10000      7.21      4.07   1.00     4.00     6.00     11.00      15.00
POLICY_AGE_AT_LOSS_DAYS   10000  2,096.48  1,029.52 110.00 1,21

---

## 1.7 Chapter 1 Summary

| Step | Action | Tool | Output |
|---|---|---|---|
| 1.1 | Load raw CSV | `src.data_loader.load_raw()` | 10,000 × 38 DataFrame |
| 1.2 | Inspect dtypes | pandas | Column type report |
| 1.3 | Load processed | `src.data_loader.load_processed()` | 10,000 × 49 DataFrame |
| 1.4 | Quality validation | Assertions vs SQL audit | All checks ✓ |
| 1.5 | Feature inventory | `src.features.get_feature_summary()` | Table 1 |
| 1.6 | Descriptive stats | pandas describe() | Tables 2 & 3 |

**The processed `df` object from this notebook is the foundation for Chapters 2–6.**
Each subsequent notebook calls `load_processed()` independently to get its own copy.

---
